# EEG2DMN_UnifiedLOSO_v1
**One pipeline, one log, every number in the paper.**

This notebook retrains and evaluates all six models of the paper on exactly the same data:

| Key | Model |
|---|---|
| `hrf_alpha` | HRF-weighted posterior alpha power (classical, no training) |
| `ridge` | Ridge regression on the flattened window (alpha chosen by inner cross-validation) |
| `mlp` | 2-layer MLP |
| `transformer_nocon` | Transformer, MSE loss only (ablation) |
| `transformer_supcon` | Transformer, MSE + Supervised Contrastive loss (proposed) |
| `labram_scratch` | LaBraM-style patch Transformer, no pretraining |

**Run order:** Cell 1 (settings) → Cell 2 → … → Cell 9. Every cell prints what it did.
Outputs go to `OUT_DIR` (on Colab: `MyDrive/EEG2DMN_UnifiedLOSO_v1_outputs`).
The notebook can be **resumed**: finished folds are saved and skipped on a re-run.

## Cell 1: Settings
Change only this cell. `SMOKE_TEST = True` runs a 2-fold, 2-epoch check in a few minutes.
Use it once first, then set it back to `False` for the real run.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # needed for deterministic CUDA

SMOKE_TEST = False

DATA_ROOT = ""          # folder that contains sub-01 ... sub-22; leave "" to auto-detect
OUT_DIR = ""            # leave "" -> Google Drive (Colab) or /kaggle/working (Kaggle)

SEEDS = [0, 1, 2]       # every deep model is trained once per seed and per fold
MODELS = ["hrf_alpha", "ridge", "mlp", "transformer_nocon", "transformer_supcon", "labram_scratch"]

# --- data ---
TR = 2.1                # fMRI repetition time (s)
WINDOW = 10             # EEG volumes per input window (21 s)
LAG_TRS = 3             # target = DMN at (last EEG volume + 3 TRs) = 6.3 s later
BANDS = [("delta", 0.5, 4.0), ("theta", 4.0, 8.0), ("alpha", 8.0, 13.0),
         ("beta", 13.0, 30.0), ("gamma", 30.0, 45.0)]
SFREQ_EXPECTED = 250
# Schaefer-2018, 100 parcels, 7 networks, 0-indexed rows of the atlas TSV
DMN_IDX = list(range(37, 50)) + list(range(89, 100))    # 24 Default-network parcels
FPN_IDX = list(range(33, 37)) + list(range(80, 89))     # 13 Control-network parcels
POSTERIOR = ["P3", "P4", "O1", "O2", "P7", "P8", "Pz", "Oz", "POz",
             "P1", "P2", "PO3", "PO4", "P5", "P6", "PO7", "PO8"]
EXACT_TOL = 0.10        # a cut is "exactly bridged" if its length is within 0.10 TR of a whole TR
MIN_EXACT_WINDOWS = 20  # r on exactly-aligned windows is reported only if a subject has >= 20

# --- training (identical for every deep model) ---
N_BINS = 5
BATCH = 64
LR = 1e-3
WD = 1e-4
MAX_EPOCHS = 50
PATIENCE = 10
N_VAL_SUBJECTS = 3      # early stopping uses 3 held-out TRAINING subjects, never the test subject
D_MODEL, N_HEADS, N_LAYERS, D_FF, DROPOUT = 128, 4, 4, 256, 0.1
LAMBDA_SUPCON, TAU = 0.5, 0.1
RIDGE_ALPHAS = [10.0 ** k for k in range(-1, 6)]

if SMOKE_TEST:
    SEEDS, MAX_EPOCHS = [0], 2
print("Settings loaded. SMOKE_TEST =", SMOKE_TEST)

## Cell 2: Connect Google Drive (Colab only) and install two libraries
On Colab a pop-up asks you to allow Drive access; accept it.
On Kaggle, turn **Internet ON** in the notebook settings.

In [ ]:
import importlib, subprocess, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
for pkg in ["mne", "nilearn"]:
    try:
        importlib.import_module(pkg)
        print(f"{pkg}: already installed")
    except ImportError:
        print(f"{pkg}: installing ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
print("Done.")

## Cell 3: Imports, reproducibility, folders, environment log

In [ ]:
import glob, json, time, copy, random, platform, warnings
import numpy as np
import pandas as pd
import scipy, scipy.signal, scipy.stats
import sklearn
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import mne
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", category=RuntimeWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_data_root():
    """Return the folder that holds sub-01/ses-01/eeg/sub-01_ses-01_task-rest_eeg.set."""
    probe = os.path.join("sub-01", "ses-01", "eeg", "sub-01_ses-01_task-rest_eeg.set")
    bases = ["/content/drive/MyDrive/Research_Dataset/Downloads", "/content/drive/MyDrive", "/kaggle/input",
             os.path.join(os.getcwd(), "NATVIEW", "Downloads"), os.getcwd()]
    for base in bases:
        if not os.path.isdir(base):
            continue
        for root, dirs, _ in os.walk(base):
            if root[len(base):].count(os.sep) >= 6:
                dirs[:] = []
                continue
            if os.path.isfile(os.path.join(root, probe)):
                return root
    raise FileNotFoundError("Could not find sub-01/ses-01/eeg/sub-01_ses-01_task-rest_eeg.set. "
                            "Set DATA_ROOT in Cell 1.")


DATA_ROOT = DATA_ROOT or find_data_root()
if not OUT_DIR:
    if os.path.isdir("/content/drive/MyDrive"):
        OUT_DIR = "/content/drive/MyDrive"          # on Drive, so a disconnect loses nothing
    elif os.path.isdir("/kaggle/working"):
        OUT_DIR = "/kaggle/working"
    else:
        OUT_DIR = os.getcwd()
    OUT_DIR = os.path.join(OUT_DIR, "EEG2DMN_UnifiedLOSO_v1_outputs" + ("_SMOKE" if SMOKE_TEST else ""))
CACHE_DIR = os.path.join(OUT_DIR, "cache")
RUNS_DIR = os.path.join(OUT_DIR, "runs")
RES_DIR = os.path.join(OUT_DIR, "results")
FIG_DIR = os.path.join(OUT_DIR, "figures")
for d in (CACHE_DIR, RUNS_DIR, RES_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

env = {
    "python": platform.python_version(), "numpy": np.__version__, "scipy": scipy.__version__,
    "sklearn": sklearn.__version__, "mne": mne.__version__, "torch": torch.__version__,
    "cuda": torch.version.cuda, "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "data_root": DATA_ROOT, "smoke_test": SMOKE_TEST,
}
with open(os.path.join(RES_DIR, "environment.json"), "w") as fh:
    json.dump(env, fh, indent=2)
print(json.dumps(env, indent=2))

## Cell 4: Find sessions and fix the channel list
* A session is used only if it has the EEG `.set`, `events.tsv`, `channels.tsv` and the
  Schaefer-100 BOLD TSV (`desc-sm0`, no global signal regression).
* Sessions have different numbers of EEG channels because bad channels were removed by the
  dataset providers. All sessions are mapped onto **one channel list** (the union of all
  channel names); a channel missing from a session is filled by spherical-spline interpolation
  (Cell 5). Features therefore mean the same electrode for every subject.

In [ ]:
def session_paths(sub, ses):
    eeg = os.path.join(DATA_ROOT, sub, ses, "eeg", f"{sub}_{ses}_task-rest_")
    atlas = os.path.join(DATA_ROOT, sub, ses, "func", f"{sub}_{ses}_task-rest_bold", "func_atlas",
                         f"{sub}_{ses}_task-rest_space-MNI152Lin_res-3mm_atlas-Schaefer2018_"
                         f"dens-100parcels7networks_desc-")
    return {"sub": sub, "ses": ses, "set": eeg + "eeg.set", "events": eeg + "events.tsv",
            "channels": eeg + "channels.tsv", "bold": atlas + "sm0_bold.tsv",
            "bold_gsr": atlas + "sm0gsr_bold.tsv"}


SESSIONS, skipped = [], []
for sub_dir in sorted(glob.glob(os.path.join(DATA_ROOT, "sub-*"))):
    sub = os.path.basename(sub_dir)
    for ses_dir in sorted(glob.glob(os.path.join(sub_dir, "ses-*"))):
        p = session_paths(sub, os.path.basename(ses_dir))
        need = ["set", "events", "channels", "bold"]
        if all(os.path.isfile(p[k]) for k in need):
            SESSIONS.append(p)
        else:
            skipped.append((sub, p["ses"], [k for k in need if not os.path.isfile(p[k])]))

SUBJECTS = sorted({s["sub"] for s in SESSIONS})
print(f"Usable sessions: {len(SESSIONS)}   subjects: {len(SUBJECTS)}")
for s in skipped:
    print("  skipped", s)
if not SESSIONS:
    raise RuntimeError("No usable sessions found. Check DATA_ROOT.")

# union of EEG channel names, ordered as in the session with the most channels
ch_lists = []
for s in SESSIONS:
    t = pd.read_csv(s["channels"], sep="\t")
    ch_lists.append([n for n, ty in zip(t["name"], t["type"]) if str(ty).lower() == "eeg"])
ref = max(ch_lists, key=len)
CANON = list(ref) + sorted({c for l in ch_lists for c in l} - set(ref))
N_CH = len(CANON)
N_INPUT = N_CH * len(BANDS)

montage = mne.channels.make_standard_montage("standard_1005")
mont_lower = {n.lower() for n in montage.ch_names}
not_in_montage = [c for c in CANON if c.lower() not in mont_lower]
if not_in_montage:
    raise RuntimeError(f"Channels without standard positions: {not_in_montage}")
POST_IDX = [CANON.index(c) for c in POSTERIOR if c in CANON]
print(f"Channel list: {N_CH} channels -> {N_INPUT} features per volume")
print("Channels per session: min", min(map(len, ch_lists)), " max", max(map(len, ch_lists)))
print("Posterior channels used by hrf_alpha:", [CANON[i] for i in POST_IDX])

# optional: confirm DMN/FPN indices against the official atlas labels (needs internet)
try:
    from nilearn import datasets as nl_datasets
    labels = [l.decode() if isinstance(l, bytes) else str(l)
              for l in nl_datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7).labels]
    labels = [l for l in labels if l.lower() != "background"]
    dmn_ok = [i for i, l in enumerate(labels) if "_Default_" in l] == DMN_IDX
    fpn_ok = [i for i, l in enumerate(labels) if "_Cont_" in l] == FPN_IDX
    print(f"Atlas label check: DMN indices match = {dmn_ok}, FPN indices match = {fpn_ok}")
    if not (dmn_ok and fpn_ok):
        raise RuntimeError("DMN/FPN indices do not match the Schaefer labels. Stop and check.")
    ATLAS_CHECK = "passed"
except RuntimeError:
    raise
except Exception as e:
    ATLAS_CHECK = f"not run ({type(e).__name__})"
    print("Atlas label check could not run:", ATLAS_CHECK)

## Cell 5: Preprocess every session (cached)
For each session:
1. **fMRI target.** Read the 100×T parcel matrix, average the 24 DMN parcels, remove a
   linear trend, z-score. The same is done for FPN and for the GSR files (GSR is used
   **only** in the DMN–FPN check of Cell 8, never for training).
2. **EEG ↔ fMRI alignment.** The preprocessed EEG has cut-out segments (`boundary` events),
   so time cannot be assumed continuous. Each fMRI volume is located by its scanner marker
   (`R128`). Consecutive markers 2.1 s apart are consecutive volumes. Across a cut, the number
   of skipped volumes is (gap + removed duration)/TR rounded to the nearest integer.
   A volume is flagged **exact** if every cut before it was bridged within `EXACT_TOL`.
   The first marker is taken as volume 1; the end-of-scan check in the log tests this.
3. **EEG features.** Missing channels are interpolated; for each volume whose 2.1-s EEG
   segment contains no cut, Welch log-power in 5 bands × all channels is computed and
   z-scored within the session.

In [ ]:
def load_parcels(path, n_parcels=100):
    with open(path) as fh:
        vals = []
        for tok in fh.read().split():
            try:
                vals.append(float(tok))
            except ValueError:
                pass                                # skip header text, if any
    if len(vals) % n_parcels:
        raise ValueError(f"{path}: {len(vals)} values is not a multiple of {n_parcels}")
    return np.asarray(vals, dtype=np.float64).reshape(n_parcels, -1)


def zdetrend(x):
    x = scipy.signal.detrend(np.asarray(x, dtype=np.float64))
    sd = x.std()
    return (x - x.mean()) / sd if sd > 1e-12 else np.zeros_like(x)


def align_volumes(events, n_vols):
    """Map fMRI volume index -> (EEG onset in s, exact flag) using R128 markers."""
    val = events["value"].astype(str).str.strip()
    dur = pd.to_numeric(events["duration"], errors="coerce").fillna(0.0).to_numpy()
    on = pd.to_numeric(events["onset"], errors="coerce").to_numpy()
    bnd = [(o, d) for o, d, v in zip(on, dur, val) if v == "boundary"]
    marks = []
    for t in np.sort(on[(val == "R128").to_numpy()]):
        if not marks or t - marks[-1] > 0.05:          # the same pulse can be logged twice
            marks.append(float(t))
    stats = {"n_markers": len(marks), "n_cuts_bridged": 0, "n_cuts_inexact": 0, "stopped_early": False}
    if not marks:
        return {}, bnd, stats
    chains, cur = [], [marks[0]]
    for a, b in zip(marks, marks[1:]):
        if abs(b - a - TR) < 0.01:
            cur.append(b)
        else:
            chains.append(cur)
            cur = [b]
    chains.append(cur)
    vols, idx, prev_end, exact = {}, 0, None, True
    for c in chains:
        if prev_end is not None:
            removed = sum(d for o, d in bnd if prev_end < o <= c[0])
            x = (c[0] - prev_end + removed) / TR
            k = int(round(x))
            if k < 1:
                stats["stopped_early"] = True
                break
            stats["n_cuts_bridged"] += 1
            if abs(x - k) > EXACT_TOL:
                exact = False
                stats["n_cuts_inexact"] += 1
            idx += k
        for i, t in enumerate(c):
            if idx + i < n_vols:
                vols[idx + i] = (t, exact)
        idx += len(c) - 1
        prev_end = c[-1]
    stats["last_marker_vol"] = int(idx)
    stats["last_marker_s"] = float(prev_end)
    return vols, bnd, stats


def eeg_on_canon(raw):
    """Return (N_CH x samples) data on CANON channels; missing channels interpolated."""
    present = {c.lower(): i for i, c in enumerate(raw.ch_names)}
    data = raw.get_data()
    full = np.zeros((N_CH, data.shape[1]))
    missing = []
    for j, c in enumerate(CANON):
        if c.lower() in present:
            full[j] = data[present[c.lower()]]
        else:
            missing.append(c)
    if missing:
        ra = mne.io.RawArray(full, mne.create_info(CANON, raw.info["sfreq"], "eeg"), verbose="ERROR")
        ra.set_montage(montage, match_case=False, on_missing="raise")
        ra.info["bads"] = list(missing)
        ra.interpolate_bads(reset_bads=True, verbose="ERROR")
        full = ra.get_data()
    return full, missing


def bandpower(seg, sf):
    f, p = scipy.signal.welch(seg, fs=sf, nperseg=int(sf), noverlap=int(sf) // 2, axis=-1)
    return np.stack([np.log10(p[:, (f >= lo) & (f < hi)].mean(axis=1) + 1e-30)
                     for _, lo, hi in BANDS], axis=1)                      # (N_CH, 5)


def process_session(s):
    out = os.path.join(CACHE_DIR, f"{s['sub']}_{s['ses']}.npz")
    if os.path.isfile(out):
        return out
    P = load_parcels(s["bold"])
    n_vols = P.shape[1]
    dmn, fpn = zdetrend(P[DMN_IDX].mean(0)), zdetrend(P[FPN_IDX].mean(0))
    dmn_parcels = np.stack([zdetrend(P[i]) for i in DMN_IDX])
    if os.path.isfile(s["bold_gsr"]):
        G = load_parcels(s["bold_gsr"])
        dmn_g, fpn_g = zdetrend(G[DMN_IDX].mean(0)), zdetrend(G[FPN_IDX].mean(0))
    else:
        dmn_g = fpn_g = np.full(n_vols, np.nan)

    raw = mne.io.read_raw_eeglab(s["set"], preload=True, verbose="ERROR")
    raw.pick("eeg")
    sf = float(raw.info["sfreq"])
    if abs(sf - SFREQ_EXPECTED) > 1e-6:
        raise ValueError(f"{s['set']}: sampling rate {sf}, expected {SFREQ_EXPECTED}")
    eeg, missing = eeg_on_canon(raw)
    n_samp, seg_len = eeg.shape[1], int(round(TR * sf))

    vols, bnd, st = align_volumes(pd.read_csv(s["events"], sep="\t"), n_vols)
    keep_v, keep_exact, feats = [], [], []
    for v in sorted(vols):
        t0, ex = vols[v]
        a = int(round(t0 * sf))
        if a < 0 or a + seg_len > n_samp:
            continue
        if any(t0 < o < t0 + TR for o, _ in bnd):        # segment crosses a cut
            continue
        feats.append(bandpower(eeg[:, a:a + seg_len], sf))
        keep_v.append(v)
        keep_exact.append(ex)
    feats = np.asarray(feats, dtype=np.float64)
    if len(feats) > 1:                                   # z-score within session
        mu, sd = feats.mean(0), feats.std(0)
        feats = (feats - mu) / np.where(sd > 1e-12, sd, 1.0)
    st.update({"n_vols_fmri": int(n_vols), "n_vols_placed": len(vols), "n_vols_good": len(keep_v),
               "n_channels_recorded": int(raw.info["nchan"]), "n_channels_interpolated": len(missing),
               "eeg_duration_s": float(n_samp / sf)})
    np.savez_compressed(out, vols=np.asarray(keep_v, dtype=np.int64), exact=np.asarray(keep_exact, bool),
                        feats=feats.astype(np.float32), dmn=dmn, fpn=fpn, dmn_gsr=dmn_g, fpn_gsr=fpn_g,
                        dmn_parcels=dmn_parcels, stats=json.dumps(st), missing=json.dumps(missing))
    return out


t_all = time.time()
for i, s in enumerate(SESSIONS, 1):
    t0 = time.time()
    process_session(s)
    st = json.loads(str(np.load(os.path.join(CACHE_DIR, f"{s['sub']}_{s['ses']}.npz"))["stats"]))
    print(f"[{i:2d}/{len(SESSIONS)}] {s['sub']} {s['ses']}: markers={st['n_markers']} "
          f"placed={st['n_vols_placed']} good={st['n_vols_good']} inexact_cuts={st['n_cuts_inexact']} "
          f"interp_ch={st['n_channels_interpolated']}  ({time.time() - t0:.1f}s)", flush=True)
print(f"Preprocessing finished in {time.time() - t_all:.0f}s")

## Cell 6: Build windows
A window = 10 consecutive good EEG volumes (v-9 … v). Target = DMN at volume v+3.
A window is **exact** if all its 10 volumes are exact.

In [ ]:
X_list, y_list, sub_list, ses_list, tv_list, ex_list = [], [], [], [], [], []
data_summary = {"sessions": {}}
for s in SESSIONS:
    d = np.load(os.path.join(CACHE_DIR, f"{s['sub']}_{s['ses']}.npz"))
    vols, exact, feats, dmn = d["vols"], d["exact"], d["feats"], d["dmn"]
    pos = {int(v): i for i, v in enumerate(vols)}
    n_win = n_exact = 0
    for v in vols:
        v = int(v)
        idx = [pos.get(v - j) for j in range(WINDOW - 1, -1, -1)]
        if any(i is None for i in idx) or v + LAG_TRS >= len(dmn):
            continue
        X_list.append(feats[idx].reshape(WINDOW, -1))
        y_list.append(dmn[v + LAG_TRS])
        sub_list.append(s["sub"])
        ses_list.append(s["ses"])
        tv_list.append(v + LAG_TRS)
        ex = bool(exact[idx].all())
        ex_list.append(ex)
        n_win += 1
        n_exact += ex
    st = json.loads(str(d["stats"]))
    st.update({"n_windows": n_win, "n_windows_exact": int(n_exact)})
    data_summary["sessions"][f"{s['sub']}_{s['ses']}"] = st

X_ALL = np.asarray(X_list, dtype=np.float32)            # (N, 10, N_INPUT)
Y_ALL = np.asarray(y_list, dtype=np.float32)
SUB_ALL = np.asarray(sub_list)
SES_ALL = np.asarray(ses_list)
TV_ALL = np.asarray(tv_list, dtype=np.int64)
EX_ALL = np.asarray(ex_list, dtype=bool)

per_sub = pd.Series(SUB_ALL).value_counts().reindex(SUBJECTS).fillna(0).astype(int)
data_summary.update({
    "n_sessions": len(SESSIONS), "n_subjects": len(SUBJECTS), "n_channels": N_CH,
    "n_features_per_volume": N_INPUT, "atlas_label_check": ATLAS_CHECK,
    "n_windows_total": int(len(Y_ALL)), "n_windows_exact": int(EX_ALL.sum()),
    "windows_per_subject": {k: int(v) for k, v in per_sub.items()},
    "sessions_with_channels_interpolated": int(sum(v["n_channels_interpolated"] > 0
                                                   for v in data_summary["sessions"].values())),
})
with open(os.path.join(RES_DIR, "data_summary.json"), "w") as fh:
    json.dump(data_summary, fh, indent=2)
print(f"Windows: {len(Y_ALL)} total, {EX_ALL.sum()} exactly aligned")
print("Windows per subject:", {k: int(v) for k, v in per_sub.items()})
if (per_sub == 0).any():
    raise RuntimeError(f"Subjects with no windows: {list(per_sub[per_sub == 0].index)}")

## Cell 7: Models and training code

In [ ]:
class TransformerDMN(nn.Module):
    """Linear input projection + Transformer encoder + attention pooling, two heads."""

    def __init__(self, n_input):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(n_input, D_MODEL), nn.LayerNorm(D_MODEL),
                                        nn.GELU(), nn.Dropout(DROPOUT))
        self.pos_embed = nn.Parameter(torch.randn(1, WINDOW, D_MODEL) * 0.02)
        layer = nn.TransformerEncoderLayer(D_MODEL, N_HEADS, D_FF, DROPOUT, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, N_LAYERS, enable_nested_tensor=False)
        self.attn_query = nn.Linear(D_MODEL, 1)
        self.proj_head = nn.Sequential(nn.Linear(D_MODEL, 64), nn.BatchNorm1d(64),
                                       nn.ReLU(inplace=True), nn.Linear(64, 32))
        self.reg_head = nn.Sequential(nn.Linear(D_MODEL, 64), nn.GELU(), nn.Dropout(DROPOUT),
                                      nn.Linear(64, 1))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        h = self.transformer(self.input_proj(x) + self.pos_embed[:, :x.size(1)])
        w = F.softmax(self.attn_query(h).squeeze(-1), dim=-1).unsqueeze(-1)
        z = (h * w).sum(1)
        return z, self.proj_head(z), self.reg_head(z).squeeze(-1)


class MLP(nn.Module):
    def __init__(self, n_input):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(WINDOW * n_input, 256), nn.ReLU(inplace=True), nn.Dropout(0.2),
                                 nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(0.2),
                                 nn.Linear(128, 1))

    def forward(self, x):
        return None, None, self.net(x.flatten(1)).squeeze(-1)


class LaBraMScratch(nn.Module):
    """LaBraM-style spatio-temporal patch Transformer: one token per (volume, channel)."""

    def __init__(self, n_ch, n_bands):
        super().__init__()
        self.n_ch = n_ch
        self.channel_proj = nn.Linear(n_bands, D_MODEL)
        self.spatial_pos = nn.Parameter(torch.randn(1, 1, n_ch, D_MODEL) * 0.02)
        self.temporal_pos = nn.Parameter(torch.randn(1, WINDOW, 1, D_MODEL) * 0.02)
        layer = nn.TransformerEncoderLayer(D_MODEL, N_HEADS, D_FF, DROPOUT, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, N_LAYERS, enable_nested_tensor=False)
        self.attn = nn.Linear(D_MODEL, 1)
        self.reg_head = nn.Sequential(nn.Linear(D_MODEL, 64), nn.GELU(), nn.Dropout(DROPOUT),
                                      nn.Linear(64, 1))

    def forward(self, x):
        b, w, _ = x.shape
        t = self.channel_proj(x.reshape(b, w, self.n_ch, -1)) + self.spatial_pos + self.temporal_pos[:, :w]
        h = self.transformer(t.reshape(b, w * self.n_ch, D_MODEL))
        a = F.softmax(self.attn(h).squeeze(-1), dim=-1).unsqueeze(-1)
        z = (h * a).sum(1)
        return z, None, self.reg_head(z).squeeze(-1)


def supcon_loss(features, labels, tau=TAU):
    """Supervised contrastive loss (Khosla et al., 2020) on L2-normalised projections."""
    b = features.size(0)
    f = F.normalize(features, dim=1)
    sim = f @ f.t() / tau
    eye = torch.eye(b, dtype=torch.bool, device=f.device)
    pos = (labels[:, None] == labels[None, :]) & ~eye
    n_pos = pos.sum(1)
    if (n_pos > 0).sum() == 0:
        return sim.sum() * 0.0
    logits = sim - sim.detach().max(1, keepdim=True).values
    log_den = torch.log((torch.exp(logits) * (~eye)).sum(1, keepdim=True) + 1e-12)
    mean_pos = ((logits - log_den) * pos).sum(1) / n_pos.clamp(min=1)
    return -mean_pos[n_pos > 0].mean()


def build_model(name):
    if name in ("transformer_nocon", "transformer_supcon"):
        return TransformerDMN(N_INPUT)
    if name == "mlp":
        return MLP(N_INPUT)
    if name == "labram_scratch":
        return LaBraMScratch(N_CH, len(BANDS))
    raise ValueError(name)


def n_params(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


@torch.no_grad()
def predict(model, X):
    model.eval()
    out = []
    for i in range(0, len(X), 256):
        out.append(model(torch.from_numpy(X[i:i + 256]).to(DEVICE))[2].float().cpu().numpy())
    return np.concatenate(out)


def train_deep(name, Xtr, ytr, Xva, yva, seed):
    set_seed(seed)
    model = build_model(name).to(DEVICE)
    edges = np.quantile(ytr, np.linspace(0, 1, N_BINS + 1))[1:-1]
    bins = np.digitize(ytr, edges).astype(np.int64)
    gen = torch.Generator().manual_seed(seed)
    dl = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr), torch.from_numpy(bins)),
                    batch_size=BATCH, shuffle=True, drop_last=True, generator=gen, num_workers=0)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)
    best, best_state, best_ep, bad, log = np.inf, None, 0, 0, []
    for ep in range(1, MAX_EPOCHS + 1):
        model.train()
        tot, n = 0.0, 0
        for xb, yb, bb in dl:
            xb, yb, bb = xb.to(DEVICE), yb.to(DEVICE), bb.to(DEVICE)
            _, proj, pred = model(xb)
            loss = F.mse_loss(pred, yb)
            if name == "transformer_supcon":
                loss = loss + LAMBDA_SUPCON * supcon_loss(proj, bb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item() * len(xb)
            n += len(xb)
        sched.step()
        val = float(np.mean((predict(model, Xva) - yva) ** 2))
        log.append({"epoch": ep, "train_loss": tot / max(n, 1), "val_mse": val})
        if val < best:
            best, best_ep, bad = val, ep, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, log, best_ep


def canonical_hrf(t):
    """SPM double-gamma haemodynamic response, peak ~5 s, undershoot ~15 s."""
    return scipy.stats.gamma.pdf(t, 6) - scipy.stats.gamma.pdf(t, 16) / 6.0


# hrf_alpha: EEG volume k of the window (k = 0..9, last = 9) is (LAG_TRS + 9 - k) TRs before the target
HRF_W = canonical_hrf(np.array([(LAG_TRS + WINDOW - 1 - k) * TR for k in range(WINDOW)]))
ALPHA_COLS = [ch * len(BANDS) + 2 for ch in POST_IDX]

param_counts = {m: n_params(build_model(m)) for m in MODELS if m not in ("ridge", "hrf_alpha")}
with open(os.path.join(RES_DIR, "parameter_counts.json"), "w") as fh:
    json.dump(param_counts, fh, indent=2)
print("Trainable parameters:", param_counts)

## Cell 8: Leave-One-Subject-Out training (resumable)
For each test subject: the other subjects are split into training and 3 validation subjects
(the same split for every model, fixed by seed and fold). Ridge uses all 21 training subjects
with its alpha chosen by 5-fold group cross-validation. Each finished run is saved to `runs/`;
re-running this cell skips runs that already exist.

In [ ]:
def r_of(a, b):
    return float(np.corrcoef(a, b)[0, 1]) if len(a) > 2 and a.std() > 1e-12 and b.std() > 1e-12 else np.nan


folds = SUBJECTS[:2] if SMOKE_TEST else SUBJECTS
t_start = time.time()
for name in MODELS:
    for seed in ([0] if name in ("ridge", "hrf_alpha") else SEEDS):
        for fi, test_sub in enumerate(folds):
            run_dir = os.path.join(RUNS_DIR, name, f"seed{seed}")
            os.makedirs(run_dir, exist_ok=True)
            out = os.path.join(run_dir, f"fold{fi:02d}_{test_sub}.npz")
            if os.path.isfile(out):
                continue
            t0 = time.time()
            te = SUB_ALL == test_sub
            train_subs = [s for s in SUBJECTS if s != test_sub]
            rng = np.random.RandomState(10_000 + 100 * seed + fi)
            val_subs = sorted(rng.choice(train_subs, N_VAL_SUBJECTS, replace=False).tolist())
            info = {"model": name, "seed": seed, "fold": fi, "test_subject": test_sub}
            if name == "hrf_alpha":
                y_pred = X_ALL[te][:, :, ALPHA_COLS].mean(-1) @ HRF_W
                info.update({"n_test": int(te.sum())})
            elif name == "ridge":
                tr = ~te
                pipe = make_pipeline(StandardScaler(), Ridge())
                gs = GridSearchCV(pipe, {"ridge__alpha": RIDGE_ALPHAS}, cv=GroupKFold(n_splits=5),
                                  scoring="neg_mean_squared_error")
                gs.fit(X_ALL[tr].reshape(tr.sum(), -1), Y_ALL[tr], groups=SUB_ALL[tr])
                y_pred = gs.predict(X_ALL[te].reshape(te.sum(), -1))
                info.update({"alpha": float(gs.best_params_["ridge__alpha"]),
                             "n_train": int(tr.sum()), "n_test": int(te.sum())})
            else:
                va = np.isin(SUB_ALL, val_subs)
                tr = ~te & ~va
                model, log, best_ep = train_deep(name, X_ALL[tr], Y_ALL[tr], X_ALL[va], Y_ALL[va], seed)
                y_pred = predict(model, X_ALL[te])
                pd.DataFrame(log).to_csv(out.replace(".npz", "_log.csv"), index=False)
                info.update({"val_subjects": val_subs, "best_epoch": best_ep, "epochs_run": len(log),
                             "n_train": int(tr.sum()), "n_val": int(va.sum()), "n_test": int(te.sum())})
            y_true = Y_ALL[te]
            ex = EX_ALL[te]
            info.update({"pearson_r": r_of(y_pred, y_true),
                         "mse": float(np.mean((y_pred - y_true) ** 2)) if name != "hrf_alpha" else None,
                         "n_test_exact": int(ex.sum()),
                         "pearson_r_exact": r_of(y_pred[ex], y_true[ex]) if ex.sum() >= MIN_EXACT_WINDOWS else None,
                         "seconds": round(time.time() - t0, 1)})
            np.savez_compressed(out, y_true=y_true, y_pred=np.asarray(y_pred, np.float32), exact=ex,
                                session=SES_ALL[te], target_vol=TV_ALL[te], info=json.dumps(info))
            print(f"{name:19s} seed{seed} fold{fi:02d} {test_sub}: r={info['pearson_r']:+.4f}  "
                  f"({info['seconds']}s, total {(time.time() - t_start) / 60:.1f} min)", flush=True)
print("All runs finished.")

## Cell 9: Results, statistics, figures, DMN–FPN check
Everything the paper reports is written to `results/` and `figures/`.
* Per-subject r = mean over seeds. Table: mean ± SD (sample SD) over subjects.
* Paired t-test and Wilcoxon signed-rank test of every model vs `transformer_supcon`.

In [ ]:
rows = []
for f in glob.glob(os.path.join(RUNS_DIR, "*", "seed*", "fold*.npz")):
    rows.append(json.loads(str(np.load(f)["info"])))
runs = pd.DataFrame(rows).sort_values(["model", "seed", "fold"])
for col in ("pearson_r", "pearson_r_exact", "mse"):
    runs[col] = pd.to_numeric(runs[col], errors="coerce")
runs.to_csv(os.path.join(RES_DIR, "all_runs.csv"), index=False)

order = [m for m in MODELS if m in set(runs["model"])]
per_sub = runs.groupby(["model", "test_subject"])["pearson_r"].mean().unstack(0)[order]
per_sub_ex = runs.groupby(["model", "test_subject"])["pearson_r_exact"].mean().unstack(0).reindex(columns=order)
per_sub.to_csv(os.path.join(RES_DIR, "per_subject_r.csv"))
per_sub_ex.to_csv(os.path.join(RES_DIR, "per_subject_r_exact_windows.csv"))

REF = "transformer_supcon"
summary, stats = [], {}
for m in order:
    r = per_sub[m]
    mse = runs[runs.model == m].groupby("test_subject")["mse"].mean()
    seed_means = runs[runs.model == m].groupby("seed")["pearson_r"].mean()
    row = {"model": m, "n_subjects": int(r.notna().sum()), "r_mean": r.mean(), "r_sd": r.std(ddof=1),
           "r_mean_exact_windows": per_sub_ex[m].mean(), "n_subjects_exact": int(per_sub_ex[m].notna().sum()),
           "mse_mean": mse.mean() if mse.notna().any() else np.nan,
           "seed_mean_r_min": seed_means.min(), "seed_mean_r_max": seed_means.max(),
           "n_positive": int((r > 0).sum())}
    if m != REF and REF in per_sub:
        both = per_sub[[REF, m]].dropna()
        t, p = scipy.stats.ttest_rel(both[REF], both[m])
        w, pw = scipy.stats.wilcoxon(both[REF] - both[m])
        stats[m] = {"n": len(both), "mean_diff": float((both[REF] - both[m]).mean()), "t": float(t),
                    "p_ttest": float(p), "W": float(w), "p_wilcoxon": float(pw),
                    "n_subjects_ref_better": int((both[REF] > both[m]).sum())}
        row.update({"p_vs_ref_ttest": p, "p_vs_ref_wilcoxon": pw})
    summary.append(row)
summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(RES_DIR, "summary_table.csv"), index=False)
with open(os.path.join(RES_DIR, "stats_vs_transformer_supcon.json"), "w") as fh:
    json.dump(stats, fh, indent=2)
pd.set_option("display.width", 200)
print(summary.round(4).to_string(index=False))

# DMN-FPN validation (fMRI only)
val_rows = []
for s in SESSIONS:
    d = np.load(os.path.join(CACHE_DIR, f"{s['sub']}_{s['ses']}.npz"))
    row = {"session": f"{s['sub']}_{s['ses']}", "r_dmn_fpn": r_of(d["dmn"], d["fpn"]),
           "r_dmn_fpn_gsr": r_of(d["dmn_gsr"], d["fpn_gsr"]) if np.isfinite(d["dmn_gsr"]).all() else np.nan}
    for k, i in enumerate(DMN_IDX):
        row[f"p{i}_vs_dmn"] = r_of(d["dmn_parcels"][k], d["dmn"])
        row[f"p{i}_vs_fpn"] = r_of(d["dmn_parcels"][k], d["fpn"])
    val_rows.append(row)
val = pd.DataFrame(val_rows)
val.to_csv(os.path.join(RES_DIR, "dmn_fpn_per_session.csv"), index=False)
reg_dmn = [val[f"p{i}_vs_dmn"].mean() for i in DMN_IDX]
reg_fpn = [val[f"p{i}_vs_fpn"].mean() for i in DMN_IDX]
dmn_fpn = {}
for col in ("r_dmn_fpn", "r_dmn_fpn_gsr"):
    x = val[col].dropna()
    if len(x) > 1:
        t, p = scipy.stats.ttest_1samp(x, 0.0)
        dmn_fpn[col] = {"n": len(x), "mean": x.mean(), "sd": x.std(ddof=1), "n_negative": int((x < 0).sum()),
                        "t": float(t), "p": float(p)}
dmn_fpn["region_vs_whole_dmn_range"] = [min(reg_dmn), max(reg_dmn)]
dmn_fpn["region_vs_fpn_range"] = [min(reg_fpn), max(reg_fpn)]
with open(os.path.join(RES_DIR, "dmn_fpn_validation.json"), "w") as fh:
    json.dump(dmn_fpn, fh, indent=2, default=float)
print(json.dumps(dmn_fpn, indent=2, default=float))

# Figures
labels = {"hrf_alpha": "HRF-alpha", "ridge": "Ridge", "mlp": "MLP", "transformer_nocon": "Transformer\n(no SupCon)",
          "transformer_supcon": "Transformer\n+ SupCon", "labram_scratch": "LaBraM\n(scratch)"}

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(len(order)), summary["r_mean"], yerr=summary["r_sd"], capsize=5, color="#4C72B0")
ax.set_xticks(range(len(order)))
ax.set_xticklabels([labels[m] for m in order])
ax.axhline(0, color="k", lw=0.6)
ax.set_ylabel("Pearson r (mean ± SD over subjects)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "model_comparison.png"), dpi=200)
plt.close(fig)

if REF in per_sub:
    fig, ax = plt.subplots(figsize=(12, 4))
    r = per_sub[REF]
    ax.bar(range(len(r)), r, color=["#2E7D32" if v > 0 else "#C62828" for v in r])
    ax.axhline(r.mean(), ls="--", color="#1565C0", label=f"mean r = {r.mean():.3f}")
    ax.set_xticks(range(len(r)))
    ax.set_xticklabels(r.index, rotation=45, ha="right")
    ax.set_ylabel("Pearson r")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, "per_subject_supcon.png"), dpi=200)
    plt.close(fig)

if {REF, "transformer_nocon"} <= set(per_sub.columns):
    fig, ax = plt.subplots(figsize=(12, 4))
    x = np.arange(len(per_sub))
    ax.bar(x - 0.2, per_sub[REF], 0.4, label="With SupCon")
    ax.bar(x + 0.2, per_sub["transformer_nocon"], 0.4, label="Without SupCon")
    ax.set_xticks(x)
    ax.set_xticklabels(per_sub.index, rotation=45, ha="right")
    ax.axhline(0, color="k", lw=0.6)
    ax.set_ylabel("Pearson r")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, "ablation_paired.png"), dpi=200)
    plt.close(fig)

    top = per_sub[REF].sort_values(ascending=False).index[:4]
    fig, axes = plt.subplots(len(top), 1, figsize=(12, 2.8 * len(top)))
    for ax, sub in zip(np.atleast_1d(axes), top):
        f = glob.glob(os.path.join(RUNS_DIR, REF, f"seed{SEEDS[0]}", f"fold*_{sub}.npz"))[0]
        d = np.load(f)
        ax.plot(d["y_true"], lw=1.1, label="Actual DMN (z)")
        ax.plot(d["y_pred"], lw=1.0, label="Predicted from EEG")
        ax.set_title(f"{sub}  (seed {SEEDS[0]}, r = {r_of(d['y_pred'], d['y_true']):.3f})")
        ax.legend(loc="upper right", fontsize=8)
    np.atleast_1d(axes)[-1].set_xlabel("Window index")
    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, "time_courses_top4.png"), dpi=200)
    plt.close(fig)

fig, ax = plt.subplots(figsize=(14, 3.5))
im = ax.imshow(per_sub[order].T.values, aspect="auto", cmap="RdYlGn", vmin=-0.3, vmax=0.5)
ax.set_yticks(range(len(order)))
ax.set_yticklabels([labels[m].replace("\n", " ") for m in order])
ax.set_xticks(range(len(per_sub)))
ax.set_xticklabels(per_sub.index, rotation=45, ha="right")
fig.colorbar(im, ax=ax, label="Pearson r")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "per_subject_heatmap.png"), dpi=200)
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 4))
v = val["r_dmn_fpn"].sort_values(ascending=False)
ax.bar(range(len(v)), v)
ax.axhline(v.mean(), ls="--", color="k", label=f"mean r = {v.mean():.2f}")
ax.set_xlabel("Session")
ax.set_ylabel("r (DMN, FPN), no GSR")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "dmn_fpn_validation.png"), dpi=200)
plt.close(fig)

print("\nFiles written:")
for d in (RES_DIR, FIG_DIR):
    for f in sorted(os.listdir(d)):
        print("  ", os.path.join(os.path.basename(d), f))